<a href="https://colab.research.google.com/github/palakvijaywar83-hub/PathSaathi/blob/main/PathSaathi_Gemma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

***Cell 1 (Markdown cell):***



# PathSaathi — Offline Study Buddy for Students
***Team: NeuralNagpur | Build with Gemma: Nagpur | Edge/On-Device***

***Cell 2 — Install libraries:***

In [3]:
!pip install -q transformers accelerate bitsandbytes torch

In [5]:
from huggingface_hub import login
login()

In [2]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "0"

***Cell 3 — Load Gemma model:***

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

MODEL_ID = "google/gemma-2-2b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("Loading Gemma model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
print("Model loaded successfully. PathSaathi is ready.")

Loading Gemma model...


tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Error while downloading from https://huggingface.co/google/gemma-2-2b-it/resolve/299a8560bedf22ed1c72a8a11e7dce4a7f9f51f8/model-00001-of-00002.safetensors: peer closed connection without sending complete message body (received 1268240186 bytes, expected 4988025760)
Trying to resume download...
Trying to resume download...
Error while downloading from https://huggingface.co/google/gemma-2-2b-it/resolve/299a8560bedf22ed1c72a8a11e7dce4a7f9f51f8/model-00001-of-00002.safetensors: peer closed connection without sending complete message body (received 1620156703 bytes, expected 3729734560)
Trying to resume download...
Trying to resume download...
Error while downloading from https://huggingface.co/google/gemma-2-2b-it/resolve/299a8560bedf22ed1c72a8a11e7dce4a7f9f51f8/model-00001-of-00002.safetensors: peer closed connection without sending complete message body (received 406565218 bytes, expected 2114927520)
Trying to resume download...
Trying to resume download...
Error while downloading from 

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Model loaded successfully. PathSaathi is ready.


***Cell 4 — (ask_pathsaathi function)***


In [11]:
def ask_pathsaathi(question, subject="General", language="English"):
    prompt = f"""You are PathSaathi, a friendly study buddy for a school student in rural India.
Subject: {subject}
Answer in {language}, using very simple words and short sentences (max 4-5 lines).
Explain like the student has no internet access and this may be their only source of help.

Student's question: {question}

Answer:"""

    chat = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(chat, return_tensors="pt", add_generation_prompt=True, return_dict=True).to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True
    )
    input_len = inputs["input_ids"].shape[-1]
    response = tokenizer.decode(output[0][input_len:], skip_special_tokens=True)
    return response.strip()

*Cell 5 — Practice question generator:*

In [12]:
def generate_practice_questions(topic, subject, num_questions=3):
    prompt = f"""Generate {num_questions} simple practice questions for a school student on the topic '{topic}' in {subject}.
Keep questions short and clear. Number them 1, 2, 3."""

    chat = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(chat, return_tensors="pt", add_generation_prompt=True, return_dict=True).to(model.device)

    output = model.generate(**inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
    input_len = inputs["input_ids"].shape[-1]
    response = tokenizer.decode(output[0][input_len:], skip_special_tokens=True)
    return response.strip()

***Cell 6 — Demo test 1:***

In [13]:
answer = ask_pathsaathi("What is photosynthesis?", subject="Science")
print("Q: What is photosynthesis?\n")
print("PathSaathi:", answer)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Q: What is photosynthesis?

PathSaathi: Photosynthesis is like a plant's way of making food.  It uses sunlight, water, and air to make sugar for energy! The leaves of plants use them to grow big and strong.


***Cell 7 — Demo test 2:***

In [14]:
questions = generate_practice_questions("Fractions", subject="Mathematics")
print("Practice Questions on Fractions:\n")
print(questions)

Practice Questions on Fractions:

Here are 3 simple practice questions on fractions:

1. What is 1/2? 
2. If you have a pizza cut into 8 slices and eat 3 slices, what fraction of the pizza did you eat?
3. What is the least common denominator of 1/2 and 1/3?
